# Эксперимент 1. Помогает ли дообучение YOLO на своих фото?

**Вопрос.** Модель YOLO, обученная на открытом наборе RDD2022 (дороги Японии, Индии, Чехии, Норвегии, США, Китая), лучше находит повреждения на наших дорогах, если дообучить её на наших снимках?

**Что сравниваем на одних и тех же новых фото — с участков, которых не было в обучении:**
- **M0** — YOLO, обученная только на RDD2022;
- **M1** — M0, дообученная на наших фото;
- **Canny** — классический метод (OpenCV), для сравнения «зачем нужна нейросеть».

**Как избегаем самообмана:**
1. Данные делятся **по ямам (группам), а не по фото**: все снимки одной ямы целиком попадают либо в обучение, либо в проверку. Ниже есть автоматическая проверка на утечку.
2. Тест — это **новые участки**: группы, снятые на других улицах, их номера заранее записаны в `test_groups.txt` (по одному в строке). M1 их не видит ни при обучении, ни при ранней остановке.
3. В тест входят **фото без повреждений** (тени, заплатки, пятна, разметка, мокрый асфальт, швы) — по ним считаем ложные срабатывания.
4. Все метрики даются с **95 % доверительными интервалами** (бутстреп по группам теста), главный результат — разница M1 − M0 со своим интервалом.
5. **Разбор случаев**: для каждого размеченного повреждения — нашли обе, только M1 (стало лучше), только M0 (стало хуже), обе пропустили; плюс ошибки в виде повреждения и ложные срабатывания. Картинки M0 | M1 для каждой категории сохраняются для слайда.

**Как запустить на Kaggle:** Settings → Accelerator: **GPU T4** (или P100), Internet: **On**. Add Input → датасет RDD2022 (поиск «RDD2022») и свой датасет с фото (папки `positives/`, `negatives/` и файл `test_groups.txt`, см. инструкцию по съёмке). Затем Run All. **Быстрый путь (≈ 1 ч):** добавьте через Add Input датасет с готовой M0 — файлом `m0_rdd2022.pt` (YOLOv8s, обученная на RDD2022, Япония + Индия; лежит в папке проекта `road-damage/models/`). Ноутбук найдёт его сам и не будет обучать M0, датасет RDD2022 тогда не нужен. Без этого файла ноутбук обучит M0 на RDD2022 сам (≈ 3–5 ч).

**Запуск 09.11.2026.** Вместо своих снимков — открытые снимки дорог Чехии из набора RDD2020 (разметка авторов набора, лицензия CC BY-SA 4.0); модель M0 обучена на Японии и Индии и чешских снимков не видела. Кадры набора перемешаны и без координат, поэтому группы-«участки» собраны программой prep_czech.py по сходству кадров (эмбеддинги YOLOv8s-cls, средняя связь, порог 0,08), тестовые группы (25 %) выбраны случайно до обучения и записаны в test_groups.txt. Обучение на процессоре (4 ядра), поэтому FT_EPOCHS = 30 и FT_PATIENCE = 10.

In [ ]:
# ===================== НАСТРОЙКИ =====================
from pathlib import Path

INPUT = Path('/kaggle/input')            # где Kaggle держит подключённые датасеты
WORK = Path('/kaggle/working')            # сюда сохраняются результаты
if not INPUT.exists():                    # запуск не на Kaggle: данные и результаты рядом с ноутбуком
    INPUT, WORK = Path('input').resolve(), Path('working').resolve()

OWN_DATA = None        # папка с positives/ и negatives/; None = найти автоматически в INPUT
RDD_ROOT = None        # папка RDD2022; None = найти автоматически в INPUT
M0_WEIGHTS = None      # путь к готовой M0 (.pt); None = найти m0_rdd2022.pt в INPUT, а если его нет — обучить на RDD2022

BASE_MODEL = 'yolov8s.pt'   # стартовые веса (COCO). Можно 'yolo11s.pt'
IMGSZ = 640
BATCH = 16
SEED = 0

RDD_COUNTRIES = None   # например ['Japan', 'Czech', 'Norway']; None = все страны
RDD_MAX_IMAGES = 12000 # ограничение, чтобы M0 обучалась за 2–3 ч на T4
RDD_EPOCHS = 30

TEST_GROUPS = None     # список групп теста, например ['G051', 'G052']; None = прочитать test_groups.txt из своих данных
TEST_SHARE = 0.25      # только если test_groups.txt нет: такая доля групп уйдёт в тест (случайно, хуже, чем отдельные улицы)
VAL_SHARE = 0.15       # доля обучающих групп для ранней остановки
FT_EPOCHS = 30         # дообучение M1 (запуск 09.11.2026 на процессоре: 30 вместо 60, иначе ~7 ч)
FT_PATIENCE = 10
FT_LR = 0.001

CONF = 0.25            # порог уверенности для precision/recall/F1 и ложных срабатываний
IOU_MATCH = 0.5        # рамка найдена верно при IoU >= 0.5
N_BOOT = 2000          # число повторов бутстрепа
RUN_CANNY = True       # классический метод для сравнения

CLASSES = ['D00', 'D10', 'D20', 'D40']   # продольная, поперечная трещина, сетка трещин, выбоина
CLASS_RU = {'D00': 'продольная трещина', 'D10': 'поперечная трещина',
            'D20': 'сетка трещин', 'D40': 'выбоина'}

In [ ]:
import subprocess, sys
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    import ultralytics
print('ultralytics', ultralytics.__version__)

import os, re, json, random, shutil, xml.etree.ElementTree as ET
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2, yaml
from PIL import Image, ImageOps
from ultralytics import YOLO
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

random.seed(SEED); np.random.seed(SEED)
WORK.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if M0_WEIGHTS is None:                    # готовая M0 среди подключённых датасетов
    M0_WEIGHTS = next((str(p) for p in INPUT.rglob('m0_rdd2022.pt')), None)
print('Готовая M0:', M0_WEIGHTS or 'нет, будет обучена на RDD2022')

def images_in(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.suffix.lower() in IMG_EXT)

# Имена классов из разных разметок -> наши коды
ALIASES = {'d00': 'D00', 'longitudinal crack': 'D00', 'longitudinal_crack': 'D00',
           'd10': 'D10', 'transverse crack': 'D10', 'transverse_crack': 'D10',
           'd20': 'D20', 'alligator crack': 'D20', 'alligator_crack': 'D20',
           'd40': 'D40', 'pothole': 'D40', 'potholes': 'D40', 'выбоина': 'D40', 'яма': 'D40'}
def to_class(name):
    return ALIASES.get(str(name).strip().lower())

## 1. Открытые данные RDD2022 → формат YOLO
Ноутбук понимает оба варианта датасета: оригинальный (XML-разметка Pascal VOC) и уже переведённый в YOLO (`data.yaml` + `.txt`). Берём только 4 класса: D00, D10, D20, D40.

In [ ]:
def find_rdd_root():
    if RDD_ROOT:
        return Path(RDD_ROOT)
    for p in INPUT.rglob('*'):
        if p.is_dir() and 'rdd' in p.name.lower():
            return p
    for p in INPUT.iterdir():                     # запасной вариант: любой датасет с xml-разметкой
        if next(p.rglob('*.xml'), None) is not None:
            return p
    raise FileNotFoundError('RDD2022 не найден: добавьте его через Add Input или укажите RDD_ROOT')

def collect_rdd(root):
    """Возвращает список (путь к фото, [(cls, xc, yc, w, h), ...]) в долях кадра."""
    items = []
    xmls = list(root.rglob('*.xml'))
    if xmls:                                       # оригинальный формат VOC
        img_index = {p.stem: p for p in images_in(root)}
        for x in xmls:
            img = img_index.get(x.stem)
            if img is None:
                continue
            if RDD_COUNTRIES and not any(c.lower() in str(img).lower() for c in RDD_COUNTRIES):
                continue
            r = ET.parse(x).getroot()
            W = float(r.findtext('size/width') or 0); H = float(r.findtext('size/height') or 0)
            if not W or not H:
                W, H = Image.open(img).size
            boxes = []
            for o in r.iter('object'):
                c = to_class(o.findtext('name'))
                b = o.find('bndbox')
                if c is None or b is None:
                    continue
                x1, y1, x2, y2 = (float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append((CLASSES.index(c), (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H))
            items.append((img, boxes))
        return items
    for y in root.rglob('*.yaml'):                 # уже в формате YOLO
        names = yaml.safe_load(open(y)).get('names')
        if names:
            break
    names = names if isinstance(names, dict) else dict(enumerate(names))
    remap = {int(k): CLASSES.index(to_class(v)) for k, v in names.items() if to_class(v)}
    lbl_index = {p.stem: p for p in root.rglob('*.txt')}
    for img in images_in(root):
        if RDD_COUNTRIES and not any(c.lower() in str(img).lower() for c in RDD_COUNTRIES):
            continue
        lp = lbl_index.get(img.stem)
        if lp is None:
            continue
        items.append((img, read_yolo(lp, remap)))
    return items

def read_yolo(path, remap=None):
    boxes = []
    for line in Path(path).read_text().split('\n'):
        v = line.split()
        if len(v) < 5:
            continue
        c = int(float(v[0]))
        c = remap.get(c) if remap is not None else c
        if c is None:
            continue
        if len(v) > 5:                              # полигон (Roboflow segmentation) -> рамка
            xs, ys = np.array(v[1::2], float), np.array(v[2::2], float)
            boxes.append((c, (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2, xs.max() - xs.min(), ys.max() - ys.min()))
        else:
            boxes.append((c, *map(float, v[1:5])))
    return boxes

def write_split(items, folder):
    (folder / 'images').mkdir(parents=True, exist_ok=True); (folder / 'labels').mkdir(parents=True, exist_ok=True)
    for i, (img, boxes) in enumerate(items):
        name = f'{i:06d}_{img.stem}'
        dst = folder / 'images' / (name + img.suffix.lower())
        if not dst.exists():
            os.symlink(img.resolve(), dst)
        (folder / 'labels' / (name + '.txt')).write_text(''.join(f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n' for c, x, y, w, h in boxes))

def write_yaml(folder, train='train', val='val'):
    y = folder / 'data.yaml'
    y.write_text(yaml.safe_dump({'path': str(folder), 'train': train, 'val': val, 'names': dict(enumerate(CLASSES))}, allow_unicode=True))
    return str(y)

if M0_WEIGHTS is None:
    rdd = find_rdd_root(); print('RDD2022:', rdd)
    items = collect_rdd(rdd)
    random.shuffle(items)
    items = items[:RDD_MAX_IMAGES]
    cnt = Counter(c for _, b in items for c, *_ in b)
    print(f'снимков: {len(items)}, рамок по классам:', {CLASSES[k]: v for k, v in sorted(cnt.items())})
    n_val = max(1, len(items) // 10)
    rdd_yolo = WORK / 'rdd_yolo'
    write_split(items[n_val:], rdd_yolo / 'train'); write_split(items[:n_val], rdd_yolo / 'val')
    rdd_yaml = write_yaml(rdd_yolo)

## 2. Модель M0: обучение только на RDD2022

In [ ]:
if M0_WEIGHTS is None:
    m = YOLO(BASE_MODEL)
    m.train(data=rdd_yaml, epochs=RDD_EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
            project=str(WORK / 'runs'), name='m0_rdd', exist_ok=True, plots=True)
    M0_WEIGHTS = str(WORK / 'm0_rdd2022.pt')
    shutil.copy(Path(m.trainer.save_dir) / 'weights' / 'best.pt', M0_WEIGHTS)  # папку выбирает ultralytics
print('M0:', M0_WEIGHTS)

## 3. Наши фото: группы (ямы/места) и негативы
Имя фото с повреждением: `G017_03.jpg` — группа `G017` (одна яма или один участок дороги), снимок 3. Roboflow может дописать к имени `_jpg.rf.xxxx` — это не мешает.
Имя фото без повреждений: `neg_shadow_012.jpg` — тип `shadow`. Разметка для них не нужна.

In [ ]:
def find_own_root():
    if OWN_DATA:
        return Path(OWN_DATA)
    for p in INPUT.rglob('positives'):
        if p.is_dir():
            return p.parent
    raise FileNotFoundError('Не найдена папка positives/: добавьте свой датасет через Add Input или укажите OWN_DATA')

own = find_own_root(); print('Свои данные:', own)
pos_dir, neg_dir = own / 'positives', own / 'negatives'

# классы в своей разметке: из data.yaml (Roboflow) или по умолчанию в порядке CLASSES
own_remap = None
for y in list(pos_dir.rglob('*.yaml')) + list(own.glob('*.yaml')):
    names = yaml.safe_load(open(y)).get('names')
    if names:
        names = names if isinstance(names, dict) else dict(enumerate(names))
        own_remap = {int(k): CLASSES.index(to_class(v)) for k, v in names.items() if to_class(v)}
        unknown = [v for v in names.values() if not to_class(v)]
        if unknown:
            print('ВНИМАНИЕ: классы не распознаны и пропущены:', unknown)
        break

GROUP_RE = re.compile(r'^(G\d+)_', re.I)
NEG_RE = re.compile(r'^neg_([a-z]+)_', re.I)
lbl_index = {p.stem: p for p in pos_dir.rglob('*.txt')}

pos, bad = [], []
for img in images_in(pos_dir):
    g = GROUP_RE.match(img.name)
    lp = lbl_index.get(img.stem)
    if g is None or lp is None:
        bad.append(img.name); continue
    pos.append({'img': img, 'group': g.group(1).upper(), 'boxes': read_yolo(lp, own_remap)})
neg = []
for img in images_in(neg_dir) if neg_dir.exists() else []:
    t = NEG_RE.match(img.name)
    neg.append({'img': img, 'type': t.group(1).lower() if t else 'other'})

if bad:
    print(f'Пропущено {len(bad)} фото (нет группы G### в имени или нет файла разметки):', bad[:10])
groups = sorted({p['group'] for p in pos})
cnt = Counter(CLASSES[c] for p in pos for c, *_ in p['boxes'])
print(f'Фото с повреждениями: {len(pos)} в {len(groups)} группах; рамок: {dict(cnt)}')
print(f'Фото без повреждений: {len(neg)}; по типам: {dict(Counter(n["type"] for n in neg))}')
assert len(groups) >= 6, f'Слишком мало групп ({len(groups)}): нужно хотя бы по несколько групп на обучение, остановку и тест'

## 4. Тест на новых участках, обучение и ранняя остановка; проверка утечки
Тест — группы из `test_groups.txt` (новые участки на других улицах). Остальные группы делятся по группам на обучение (≈ 85 %) и раннюю остановку (≈ 15 %). Затем проверяем: (а) ни одна группа не попала в две части; (б) нет почти одинаковых снимков (перцептивный хеш) в разных частях — это ловит ошибки в именах файлов.

In [ ]:
if TEST_GROUPS is None:
    tf = next(iter(own.rglob('test_groups.txt')), None) or next(iter(INPUT.rglob('test_groups.txt')), None)
    if tf is not None:
        TEST_GROUPS = [l.strip().upper() for l in tf.read_text(encoding='utf-8').split() if l.strip()]
        print('Тестовые группы из', tf)
if TEST_GROUPS:
    test_g = set(g.upper() for g in TEST_GROUPS)
    missing = test_g - set(groups)
    if missing:
        print('ВНИМАНИЕ: в test_groups.txt есть группы без фото:', sorted(missing))
    test_g &= set(groups)
else:
    gs = list(groups); random.Random(SEED).shuffle(gs)
    test_g = set(gs[:max(2, round(len(gs) * TEST_SHARE))])
    print('ВНИМАНИЕ: test_groups.txt не найден, тест выбран случайно. Для честной проверки снимите новые участки на других улицах.')
rest = [g for g in groups if g not in test_g]; random.Random(SEED + 1).shuffle(rest)
val_g = set(rest[:max(1, round(len(rest) * VAL_SHARE))])
part_of = {g: 'test' if g in test_g else 'val' if g in val_g else 'train' for g in groups}
for p in pos:
    p['part'] = part_of[p['group']]
assert test_g and val_g and len(test_g) + len(val_g) < len(groups), 'Не хватает групп на одну из частей'

# (а) группа целиком в одной части
g2p = defaultdict(set)
for p in pos:
    g2p[p['group']].add(p['part'])
assert all(len(v) == 1 for v in g2p.values()), 'УТЕЧКА: группа попала в несколько частей'

# (б) почти одинаковые снимки в разных частях
def dhash(path, size=8):
    im = ImageOps.exif_transpose(Image.open(path)).convert('L').resize((size + 1, size))
    a = np.asarray(im, dtype=np.int16)
    return np.packbits(a[:, 1:] > a[:, :-1])
H = np.array([dhash(p['img']) for p in pos])
dist = np.unpackbits(H[:, None, :] ^ H[None, :, :], axis=2).sum(2)
parts_arr = np.array([p['part'] for p in pos])
sus = [(pos[i]['img'].name, pos[j]['img'].name, int(dist[i, j]))
       for i, j in zip(*np.where((dist <= 5) & (parts_arr[:, None] != parts_arr[None, :])))
       if i < j]
for part, ru in [('train', 'обучение'), ('val', 'ранняя остановка'), ('test', 'тест — новые участки')]:
    ps = [p for p in pos if p['part'] == part]
    print(f'{ru}: групп {sum(1 for v in part_of.values() if v == part)}, фото {len(ps)}, рамок {sum(len(p["boxes"]) for p in ps)}')
print('Тестовые группы:', sorted(test_g))
print('Группы в нескольких частях: 0 — проверка пройдена')
print(f'Подозрительно похожих пар фото из разных частей: {len(sus)}', sus[:10])
if sus:
    print('Проверьте эти пары: скорее всего, это одна яма с разными номерами группы.')
test = [p for p in pos if p['part'] == 'test']
groups = sorted(test_g)          # дальше метрики и бутстреп — только по тестовым группам

## 5. Дообучение M1 и предсказания обеих моделей на одних и тех же тестовых фото
M1 = M0, дообученная на обучающей части (ранняя остановка — по своей части). M0 и M1 прогоняются на одних и тех же фото новых участков и на фото без повреждений. Предсказания сохраняем с низким порогом (0,001), чтобы считать mAP. Веса M1 сохраняются как `m1_finetuned.pt` — их можно положить в папку `models/` сайта.

In [ ]:
def predict(model, paths, conf=0.001):
    out = {}
    for i in range(0, len(paths), 16):
        for path, r in zip(paths[i:i + 16], model.predict([str(p) for p in paths[i:i + 16]], imgsz=IMGSZ, conf=conf, verbose=False)):
            b = r.boxes
            out[str(path)] = list(zip(b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxyn.cpu().numpy()))
    return out

tr = [(p['img'], p['boxes']) for p in pos if p['part'] == 'train']
va = [(p['img'], p['boxes']) for p in pos if p['part'] == 'val']
d = WORK / 'own_split'
shutil.rmtree(d, ignore_errors=True)
write_split(tr, d / 'train'); write_split(va, d / 'val')
m = YOLO(M0_WEIGHTS)
m.train(data=write_yaml(d), epochs=FT_EPOCHS, patience=FT_PATIENCE, imgsz=IMGSZ, batch=BATCH, seed=SEED,
        optimizer='AdamW', lr0=FT_LR, warmup_epochs=1, project=str(WORK / 'runs'), name='m1', exist_ok=True,
        plots=False, verbose=False)
M1_WEIGHTS = WORK / 'm1_finetuned.pt'
shutil.copy(Path(m.trainer.save_dir) / 'weights' / 'best.pt', M1_WEIGHTS)
print(f'обучение {len(tr)} фото, ранняя остановка {len(va)}, тест {len(test)} фото + {len(neg)} без повреждений; M1: {M1_WEIGHTS}')

test_paths = [p['img'] for p in test]; neg_paths = [n['img'] for n in neg]
preds, neg_preds = {}, {}
for name, w in [('M0', M0_WEIGHTS), ('M1', M1_WEIGHTS)]:
    model = YOLO(w)
    preds[name] = predict(model, test_paths)
    neg_preds[name] = [predict(model, neg_paths)]

## 6. Классический метод (OpenCV, Canny)
Выделяем границы, склеиваем их морфологией и берём рамки крупных контуров. Тип повреждения метод не определяет, поэтому он сравнивается с нейросетью только по факту «нашёл повреждение / нет» (без учёта класса).

In [ ]:
def canny_detect(path):
    im = cv2.imread(str(path)); h, w = im.shape[:2]
    s = 960 / max(h, w); im = cv2.resize(im, (int(w * s), int(h * s)))
    g = cv2.GaussianBlur(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    e = cv2.Canny(g, 50, 150)
    e = cv2.morphologyEx(e, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    cnts, _ = cv2.findContours(e, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H_, W_ = e.shape; out = []
    for c in cnts:
        x, y, bw, bh = cv2.boundingRect(c)
        a = bw * bh / (W_ * H_)
        if a > 0.003:
            out.append((-1, 0.5 + 0.5 * min(1.0, a * 5), np.array(  # «уверенность» >= 0.5: порог CONF не отсекает рамки Canny, нужна только для сортировки
                [x / W_, y / H_, (x + bw) / W_, (y + bh) / H_])))
    return out

if RUN_CANNY:
    preds['Canny'] = {str(p): canny_detect(p) for p in test_paths}
    neg_preds['Canny'] = [{str(p): canny_detect(p) for p in neg_paths}]

## 7. Метрики с доверительными интервалами
- **Precision** — какая доля найденных рамок верна; **Recall** — какая доля настоящих повреждений найдена; **F1** — их среднее гармоническое (при пороге уверенности CONF).
- **mAP@0,5** — средняя по классам площадь под кривой precision–recall (рамка верна при IoU ≥ 0,5).
- **Ложные срабатывания** — среднее число рамок на одно фото без повреждений.
- 95 % интервалы — бутстреп: много раз случайно выбираем тестовые группы с возвращением и пересчитываем метрику. Разница M1 − M0 считается на одних и тех же выборках (парный бутстреп).

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def gt_xyxy(boxes):
    return [(c, np.array([x - w / 2, y - h / 2, x + w / 2, y + h / 2])) for c, x, y, w, h in boxes]

def match(model_preds, agnostic):
    """Для каждой группы: список (класс, уверенность, верно?) и число настоящих рамок по классам."""
    per_group = defaultdict(lambda: {'det': [], 'gt': Counter()})
    for p in test:
        gts = gt_xyxy(p['boxes']); used = [False] * len(gts)
        G = per_group[p['group']]
        for c, *_ in gts:
            G['gt'][-1 if agnostic else c] += 1
        for c, s, box in sorted(model_preds[str(p['img'])], key=lambda d: -d[1]):
            best, bj = IOU_MATCH, -1
            for j, (gc, gb) in enumerate(gts):
                if not used[j] and (agnostic or gc == c):
                    v = iou(box, gb)
                    if v >= best:
                        best, bj = v, j
            if bj >= 0:
                used[bj] = True
            G['det'].append((-1 if agnostic else int(c), float(s), bj >= 0))
    for G in per_group.values():
        G['arr'] = np.array(G['det'], dtype=float).reshape(-1, 3)
    return dict(per_group)

def ap_voc(scores, tp, n_gt, w):
    if n_gt == 0:
        return np.nan
    o = np.argsort(-scores); tp, w = tp[o], w[o]
    ctp = np.cumsum(w * tp); cfp = np.cumsum(w * (1 - tp))
    rec = ctp / n_gt; prec = ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1]))

def metrics(per_group, weights):
    """weights: {группа: сколько раз взята в бутстреп-выборку}."""
    parts = [np.c_[G['arr'], np.full(len(G['arr']), weights[g])] for g, G in per_group.items() if weights.get(g)]
    d = np.concatenate(parts) if parts else np.zeros((0, 4))
    gt = Counter()
    for g, G in per_group.items():
        for c, n in G['gt'].items():
            gt[c] += n * weights.get(g, 0)
    keep = d[:, 1] >= CONF
    TP = (d[keep, 2] * d[keep, 3]).sum(); FP = ((1 - d[keep, 2]) * d[keep, 3]).sum(); NG = sum(gt.values())
    P = TP / (TP + FP) if TP + FP else 0.0; R = TP / NG if NG else np.nan
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    aps = {c: ap_voc(d[d[:, 0] == c, 1], d[d[:, 0] == c, 2], gt[c], d[d[:, 0] == c, 3]) for c in gt if gt[c] > 0}
    return {'P': P, 'R': R, 'F1': F1, 'mAP50': float(np.nanmean(list(aps.values()))) if aps else np.nan,
            **{f'AP50_{CLASSES[c]}': v for c, v in aps.items() if c >= 0}}

def fp_per_neg(neg_pred_list, idx):
    """Среднее число рамок на фото без повреждений."""
    return float(np.mean([[sum(s >= CONF for _, s, _ in npred[str(neg[i]['img'])]) for i in idx] for npred in neg_pred_list]))

rng = np.random.default_rng(SEED)
boot_g = [Counter(rng.choice(groups, len(groups))) for _ in range(N_BOOT)]
boot_n = [rng.integers(0, len(neg), len(neg)) for _ in range(N_BOOT)] if neg else []
all_w = Counter({g: 1 for g in groups})

def ci(v):
    v = np.array(v, float); v = v[~np.isnan(v)]
    return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) else (np.nan, np.nan)

rows, boots = [], {}
for name in preds:
    for agn in ([False, True] if name != 'Canny' else [True]):
        pg = match(preds[name], agn)
        point = metrics(pg, all_w)
        bs = [metrics(pg, w) for w in boot_g]
        key = (name, 'без учёта класса' if agn else 'с учётом класса')
        boots[key] = bs
        row = {'модель': name, 'режим': key[1]}
        for k, v in point.items():
            lo, hi = ci([b.get(k, np.nan) for b in bs])
            row[k] = f'{v:.3f} [{lo:.3f}; {hi:.3f}]'
        if neg:
            v = fp_per_neg(neg_preds[name], range(len(neg)))
            lo, hi = ci([fp_per_neg(neg_preds[name], b) for b in boot_n])
            row['ложных рамок на фото без ям'] = f'{v:.2f} [{lo:.2f}; {hi:.2f}]'
        rows.append(row)
res = pd.DataFrame(rows)
res.to_csv(WORK / 'results.csv', index=False)
pd.set_option('display.max_colwidth', None)
res

### Главный результат: насколько дообучение изменило метрики (M1 − M0)

In [ ]:
diff_rows = []
for mode in ['с учётом класса', 'без учёта класса']:
    a, b = boots[('M0', mode)], boots[('M1', mode)]
    for k in ['mAP50', 'P', 'R', 'F1']:
        d = [y[k] - x[k] for x, y in zip(a, b)]
        lo, hi = ci(d)
        p0 = metrics(match(preds['M0'], mode != 'с учётом класса'), all_w)[k]
        p1 = metrics(match(preds['M1'], mode != 'с учётом класса'), all_w)[k]
        diff_rows.append({'режим': mode, 'метрика': k, 'M0': round(p0, 3), 'M1': round(p1, 3),
                          'разница': round(p1 - p0, 3), '95% интервал': f'[{lo:+.3f}; {hi:+.3f}]',
                          'вывод': 'улучшение' if lo > 0 else ('ухудшение' if hi < 0 else 'различие не доказано')})
diff = pd.DataFrame(diff_rows)
diff.to_csv(WORK / 'finetune_effect.csv', index=False)
diff

### Ложные срабатывания по типам «обманок»

In [ ]:
if neg:
    types = sorted({n['type'] for n in neg})
    tab = pd.DataFrame({name: {t: fp_per_neg(neg_preds[name], [i for i, n in enumerate(neg) if n['type'] == t]) for t in types}
                        for name in neg_preds}).round(2)
    tab['фото'] = [sum(n['type'] == t for n in neg) for t in types]
    tab.to_csv(WORK / 'false_positives_by_type.csv')
    display(tab)

## 8. Разбор случаев: одни и те же тестовые фото, M0 и M1
Каждое размеченное повреждение на фото новых участков попадает в категорию: **нашли обе**, **только M1 — стало лучше**, **только M0 — стало хуже**, **обе пропустили**. Отдельно — **ошибка в виде повреждения** (рамка на месте, класс не тот) и **ложные срабатывания** (рамка там, где повреждения нет). Найдено = рамка модели с уверенностью ≥ CONF и IoU ≥ 0,5.

Файлы: `cases.csv` — каждое повреждение и его категория; `false_positives.csv` — каждое ложное срабатывание, графу `причина` (тень, шов, заплатка, разметка, мокрый асфальт, люк, другое) заполнить вручную по картинкам; `cases/*.png` — примеры M0 | M1 для слайда (зелёное — разметка, красное — рамки модели).

In [ ]:
def found(model_preds, p, need_class):
    """Для каждой размеченной рамки: индекс рамки модели, которая её нашла (или -1), и список лишних рамок."""
    gts = gt_xyxy(p['boxes']); hit = [-1] * len(gts); extra = []
    dets = sorted([d for d in model_preds[str(p['img'])] if d[1] >= CONF], key=lambda d: -d[1])
    for k, (c, s, box) in enumerate(dets):
        best, bj = IOU_MATCH, -1
        for j, (gc, gb) in enumerate(gts):
            if hit[j] < 0 and (not need_class or gc == c):
                v = iou(box, gb)
                if v >= best:
                    best, bj = v, j
        if bj >= 0:
            hit[bj] = k
        else:
            extra.append((c, s, box))
    return hit, dets, extra

CAT = {(True, True): 'нашли обе', (False, True): 'только M1 — стало лучше',
       (True, False): 'только M0 — стало хуже', (False, False): 'обе пропустили'}
rows, fps = [], []
for p in test:
    h0, d0, x0 = found(preds['M0'], p, need_class=False)
    h1, d1, x1 = found(preds['M1'], p, need_class=False)
    for j, (c, *_) in enumerate(p['boxes']):
        f0, f1 = h0[j] >= 0, h1[j] >= 0
        rows.append({'фото': p['img'].name, 'группа': p['group'], 'класс': CLASSES[c], 'категория': CAT[(f0, f1)],
                     'класс у M0': CLASSES[d0[h0[j]][0]] if f0 else '', 'класс у M1': CLASSES[d1[h1[j]][0]] if f1 else ''})
    for name, xs in [('M0', x0), ('M1', x1)]:
        for c, s, b in xs:
            fps.append({'модель': name, 'фото': p['img'].name, 'тип фото': 'новый участок', 'класс': CLASSES[c],
                        'уверенность': round(float(s), 2), 'причина': ''})
for n in neg:
    for name in ['M0', 'M1']:
        for c, s, b in neg_preds[name][0][str(n['img'])]:
            if s >= CONF:
                fps.append({'модель': name, 'фото': n['img'].name, 'тип фото': n['type'], 'класс': CLASSES[c],
                            'уверенность': round(float(s), 2), 'причина': n['type']})
cases = pd.DataFrame(rows); cases.to_csv(WORK / 'cases.csv', index=False)
fp = pd.DataFrame(fps, columns=['модель', 'фото', 'тип фото', 'класс', 'уверенность', 'причина'])
fp.to_csv(WORK / 'false_positives.csv', index=False)

summary = {k: int((cases['категория'] == k).sum()) for k in CAT.values()}
summary['обе пропустили, в том числе трещины'] = int(((cases['категория'] == 'обе пропустили') & cases['класс'].isin(['D00', 'D10', 'D20'])).sum())
for name in ['M0', 'M1']:
    col = f'класс у {name}'
    summary[f'ошибка в виде повреждения у {name}'] = int(((cases[col] != '') & (cases[col] != cases['класс'])).sum())
    summary[f'ложные срабатывания {name} на фото новых участков'] = int(((fp['модель'] == name) & (fp['тип фото'] == 'новый участок')).sum())
    summary[f'ложные срабатывания {name} на фото без повреждений'] = int(((fp['модель'] == name) & (fp['тип фото'] != 'новый участок')).sum())
summary = pd.Series(summary, name='число'); summary.to_csv(WORK / 'cases_summary.csv')
display(summary.to_frame())
if len(fp):
    display(fp[fp['тип фото'] != 'новый участок'].pivot_table(index='тип фото', columns='модель', values='фото', aggfunc='count', fill_value=0))

def draw(ax, path, boxes, model_preds, title):
    im = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert('RGB')); h, w = im.shape[:2]
    ax.imshow(im); ax.set_title(title, fontsize=9); ax.axis('off')
    for c, b in gt_xyxy(boxes):
        ax.add_patch(plt.Rectangle((b[0] * w, b[1] * h), (b[2] - b[0]) * w, (b[3] - b[1]) * h, fill=False, ec='lime', lw=2))
    for c, s, b in model_preds[str(path)]:
        if s >= CONF:
            ax.add_patch(plt.Rectangle((b[0] * w, b[1] * h), (b[2] - b[0]) * w, (b[3] - b[1]) * h, fill=False, ec='red', lw=1.5))
            ax.text(b[0] * w, b[1] * h, f'{CLASSES[c] if c >= 0 else ""} {s:.2f}', color='red', fontsize=7)

def gallery(items, fname, n=2):
    """items: список (путь, разметка, подпись) — по n примеров, слева M0, справа M1."""
    items = items[:n]
    if not items:
        return
    fig, axs = plt.subplots(len(items), 2, figsize=(10, 4 * len(items)), squeeze=False)
    for r, (path, boxes, note) in enumerate(items):
        draw(axs[r][0], path, boxes, preds['M0'] if path in map(str, test_paths) else neg_preds['M0'][0], f'M0 — {Path(path).name} {note}')
        draw(axs[r][1], path, boxes, preds['M1'] if path in map(str, test_paths) else neg_preds['M1'][0], 'M1 (этот участок не видела)')
    plt.tight_layout(); plt.savefig(WORK / 'cases' / fname, dpi=120); plt.show()

(WORK / 'cases').mkdir(exist_ok=True)
by_name = {p['img'].name: p for p in test}
for k, fname in [('только M1 — стало лучше', 'стало_лучше.png'), ('только M0 — стало хуже', 'стало_хуже.png'),
                 ('обе пропустили', 'обе_пропустили.png'), ('нашли обе', 'нашли_обе.png')]:
    names = list(dict.fromkeys(cases.loc[cases['категория'] == k, 'фото']))
    print(k, '—', len(names), 'фото')
    gallery([(str(by_name[nm]['img']), by_name[nm]['boxes'], '') for nm in names], fname)
for t in sorted({n['type'] for n in neg}):          # «приняли тень за трещину» и т. п.
    names = list(dict.fromkeys(fp.loc[fp['тип фото'] == t, 'фото']))
    if names:
        print('ложные срабатывания на фото типа', t, '—', len(names), 'фото')
        npath = {n['img'].name: str(n['img']) for n in neg}
        gallery([(npath[nm], [], f'({t})') for nm in names], f'ложные_{t}.png')

## 9. Итог для защиты (заполняется автоматически)
Формулировки осторожные: если интервал разницы включает 0, мы **не утверждаем** улучшение.

In [ ]:
r = diff[(diff['режим'] == 'с учётом класса') & (diff['метрика'] == 'mAP50')].iloc[0]
print(f'Тест: {len(test)} фото с повреждениями ({len(groups)} новых участков, которых не было в обучении) и {len(neg)} фото без повреждений.')
print(f'mAP@0,5: M0 = {r["M0"]:.2f}, M1 = {r["M1"]:.2f}, разница {r["разница"]:+.2f}, 95 % интервал {r["95% интервал"]} — {r["вывод"]}.')
if neg:
    print('Ложных рамок на фото без повреждений:', {k: v for k, v in zip(res['модель'] + ' / ' + res['режим'], res.get('ложных рамок на фото без ям', []))})
print('Разбор случаев:', summary.to_dict())
print('Ограничения: небольшой объём своих данных (широкие интервалы); снимки одного города и сезона; площадь и глубину по снимку не определяем.')
json.dump({'n_pos_test': len(test), 'n_test_groups': len(groups), 'test_groups': groups, 'n_neg': len(neg), 'conf': CONF,
           'base_model': BASE_MODEL, 'm0_weights': str(M0_WEIGHTS)}, open(WORK / 'run_info.json', 'w'), ensure_ascii=False, indent=1)

## 10. Дополнительно: строгий тест и время обработки
Участки определены по сходству кадров, а не по координатам, поэтому похожий участок мог попасть и в обучение. Строгий тест — только тестовые снимки, у которых ближайший снимок обучения дальше 0,06 по косинусному расстоянию эмбеддингов (файл test_novelty.csv).

In [ ]:
import time
nov = pd.read_csv(next(INPUT.rglob('test_novelty.csv')))
far = set(nov.loc[nov['min_cos_dist_to_train'] > 0.06, 'file'])
test_all, groups_all = test, groups
test = [p for p in test_all if p['img'].name in far]
groups = sorted({p['group'] for p in test})
rng2 = np.random.default_rng(SEED + 7)
bg = [Counter(rng2.choice(groups, len(groups))) for _ in range(N_BOOT)]
aw = Counter({g: 1 for g in groups})
strict = []
for mode, agn in [('с учётом класса', False), ('без учёта класса', True)]:
    pg0, pg1 = match(preds['M0'], agn), match(preds['M1'], agn)
    for k in ['mAP50', 'P', 'R', 'F1']:
        a = [metrics(pg0, w)[k] for w in bg]; b = [metrics(pg1, w)[k] for w in bg]
        lo, hi = ci([y - x for x, y in zip(a, b)])
        p0, p1 = metrics(pg0, aw)[k], metrics(pg1, aw)[k]
        strict.append({'режим': mode, 'метрика': k, 'M0': round(p0, 3), 'M1': round(p1, 3), 'разница': round(p1 - p0, 3),
                       '95% интервал': f'[{lo:+.3f}; {hi:+.3f}]', 'вывод': 'улучшение' if lo > 0 else ('ухудшение' if hi < 0 else 'различие не доказано')})
strict = pd.DataFrame(strict); strict.to_csv(WORK / 'finetune_effect_strict.csv', index=False)
print(f'строгий тест: {len(test)} фото в {len(groups)} группах')
display(strict)
test, groups = test_all, groups_all

ms = {}
for name, w in [('M0', M0_WEIGHTS), ('M1', M1_WEIGHTS)]:
    mdl = YOLO(w); mdl.predict(str(test_paths[0]), imgsz=IMGSZ, verbose=False)
    t0 = time.time()
    for pth in test_paths[:50]:
        mdl.predict(str(pth), imgsz=IMGSZ, conf=CONF, verbose=False)
    ms[name] = (time.time() - t0) / 50 * 1000
rc = pd.read_csv(Path(m.trainer.save_dir) / 'results.csv')
train_info = {'epochs_run': int(rc['epoch'].max()), 'best_epoch': int(rc.loc[rc['metrics/mAP50(B)'].idxmax(), 'epoch']),
              'train_minutes': round(float(rc['time'].max()) / 60, 1), 'ms_per_image_cpu': {k: round(v) for k, v in ms.items()},
              'n_train': len(tr), 'n_val': len(va)}
json.dump(train_info, open(WORK / 'train_info.json', 'w'), ensure_ascii=False, indent=1)
print(train_info)